In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP — Clone Repo & Install Packages
# ═══════════════════════════════════════════════════════════════

import os

# Clone repo
!git clone https://github.com/aneek22112007-tech/SocraticRL
%cd SocraticRL

print("✅ Repo cloned")

# Install packages
!pip install -q transformers torch peft bitsandbytes wandb datasets trl

print("✅ Packages installed")

# Login to HuggingFace
from huggingface_hub import login
login()

print("✅ HuggingFace login complete")

# Login to W&B
import wandb
wandb.login()

print("\n🎉 Setup complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Load Model (Qwen2.5-7B in 4-bit)
# ═══════════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("Loading Qwen2.5-7B in 4-bit...")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

print("✅ Model loaded in 4-bit")

# Add LoRA adapters
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
print("✅ LoRA adapters added")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Sanity Check — Verify Reward Function
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/SocraticRL')

from reward import compute_reward

print("Testing reward function...\n")

good = 'If you dropped a feather and a hammer in a vacuum, what would you expect to happen?'
bad  = 'The answer is that heavier objects fall faster due to gravity.'

progress_keywords = ['acceleration', 'same rate', 'air resistance', 'galileo', 'mass', 'vacuum']

r_good = compute_reward(good, 'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])
r_bad  = compute_reward(bad,  'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])

print(f'Good question reward: {r_good.reward:+.2f}  ({r_good.feedback})')
print(f'Bad  answer  reward:  {r_bad.reward:+.2f}  ({r_bad.feedback})')

assert r_good.reward > 0, 'Good question should get positive reward'
assert r_bad.reward  < 0, 'Direct answer should get negative reward'

print('\n✅ Reward function OK')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Build IMPROVED Training Dataset (50 scenarios × 3 variations)
# ═══════════════════════════════════════════════════════════════

from datasets import Dataset
from students.scenarios_expanded import TRAINING_SCENARIOS_EXPANDED

print(f"Building dataset from {len(TRAINING_SCENARIOS_EXPANDED)} scenarios...")

rows = []

for scenario in TRAINING_SCENARIOS_EXPANDED:
    # Variation 1: Direct Socratic prompt
    prompt1 = (
        f"""You are a Socratic tutor helping a student understand {scenario.topic}.

Student's current belief: {scenario.misconception}

Your role:
- Ask questions that challenge their thinking
- Never state the answer directly
- Guide them to discover the correct understanding themselves
- Be specific to the topic, not generic

What is your first Socratic question?"""
    )
    rows.append({"prompt": prompt1})

    # Variation 2: Misconception-focused
    prompt2 = (
        f"""Topic: {scenario.topic}
Student misconception: {scenario.misconception}

As a Socratic tutor, ask a question that:
1. Challenges this specific misconception
2. Doesn't give away the answer
3. Helps them think deeper

Your question:"""
    )
    rows.append({"prompt": prompt2})

    # Variation 3: Persona-based
    prompt3 = (
        f"""You are tutoring a {scenario.persona}.
They believe: {scenario.misconception}
Topic: {scenario.topic}

Ask a Socratic question that helps them reconsider their belief.
Question:"""
    )
    rows.append({"prompt": prompt3})

dataset = Dataset.from_list(rows)
print(f"✅ Dataset size: {len(dataset)} prompts")
print(f"   ({len(TRAINING_SCENARIOS_EXPANDED)} scenarios × 3 variations)")
print(f"\nExample prompt:\n{dataset[0]['prompt'][:300]}...")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Define Reward Function for GRPO
# ═══════════════════════════════════════════════════════════════

import wandb
from reward import compute_reward
from students.scenarios_expanded import TRAINING_SCENARIOS_EXPANDED

# Initialize W&B
wandb.init(project='socratic-rl-training', resume='allow')
print(f"✅ W&B initialized: {wandb.run.url}")

def socratic_reward(prompts, completions, **kwargs):
    """
    GRPO reward function.
    prompts:     list of prompt strings
    completions: list of generated strings
    Returns:     list of float rewards
    """
    rewards = []

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        # Extract topic from prompt
        topic = ''
        for line in prompt.split('\\n'):
            if 'Topic:' in line or 'topic' in line.lower():
                topic = line.split(':', 1)[-1].strip() if ':' in line else line
                break

        scenario_idx = (i // 3) % len(TRAINING_SCENARIOS_EXPANDED)
        scenario = TRAINING_SCENARIOS_EXPANDED[scenario_idx]
        question = completion.strip().splitlines()[0].strip() if completion.strip() else ""
        
        if not question or len(question) < 5:
            rewards.append(-1.0)
            continue

        # Score with reward.py
        progress_keywords = scenario.correct_answer.lower().split()
        result = compute_reward(
            question=question,
            topic=topic or scenario.topic,
            progress_keywords=progress_keywords,
            understanding_score=0.1,
            turn_count=1,
            question_history=[],
        )
        total_reward = result.reward
        rewards.append(total_reward)

    mean_reward = sum(rewards) / max(len(rewards), 1)
    wandb.log({'mean_reward': mean_reward})
    print(f'  Batch: mean_reward={mean_reward:.3f}')
    return rewards

print("✅ Reward function defined")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: TRAIN — GRPO for 3 Epochs (IMPROVED)
# ═══════════════════════════════════════════════════════════════

from trl import GRPOConfig, GRPOTrainer

print("Configuring GRPO trainer...")

training_args = GRPOConfig(
    output_dir='./socratic-agent',
    num_generations=4,
    max_completion_length=256,  # Longer completions for better questions
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,  # Slightly higher learning rate
    num_train_epochs=3,  # 3 epochs instead of 1
    logging_steps=5,
    save_steps=50,
    report_to='wandb',
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[socratic_reward],
    args=training_args,
    train_dataset=dataset,
)

print("\n🚀 Starting GRPO training...")
print(f"Dataset size: {len(dataset)} samples")
print(f"Epochs: 3")
print(f"Watch for mean_reward to climb from ~-1.5 to ~+1.0\n")

trainer.train()

print("\n✅ Training complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Save Model & Push to Hub
# ═══════════════════════════════════════════════════════════════

import os

print("Saving model...")
model.save_pretrained('socratic-agent-final')
tokenizer.save_pretrained('socratic-agent-final')
print("✅ Saved to ./socratic-agent-final")

# Push to HF Hub
HF_USERNAME = 'aneek22112007-tech'
repo_id = f'{HF_USERNAME}/socratic-rl-agent'

print(f"\nPushing to HuggingFace Hub: {repo_id}")
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f"✅ Pushed to https://huggingface.co/{repo_id}")

# Print W&B link
print(f"\n📊 W&B Dashboard: {wandb.run.url}")
print("\n✅ All done! Next: Download reward_curve_real.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Download Real Reward Curve from W&B
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import wandb

print("Downloading W&B data...")

run = wandb.run
history = run.history()

print(f"History shape: {history.shape}")
print(f"Columns: {list(history.columns)}")

if 'mean_reward' in history.columns:
    steps = history['_step'].values
    rewards = history['mean_reward'].values
    
    print(f"\nReward stats:")
    print(f"  Initial: {rewards[0]:.2f}")
    print(f"  Final:   {rewards[-1]:.2f}")
    print(f"  Improvement: {rewards[-1] - rewards[0]:+.2f}")
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(steps, rewards, linewidth=2.5, color='#1D9E75', marker='o', markersize=4)
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1.5, label='Break-even')
    plt.xlabel('Training Step', fontsize=12)
    plt.ylabel('Mean Reward', fontsize=12)
    plt.title('SocraticRL — GRPO Training on Colab T4 (Improved)', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('reward_curve_real.png', dpi=150)
    print("\n✅ Saved reward_curve_real.png")
    plt.show()
else:
    print("ERROR: 'mean_reward' not in W&B history")

# Download
from google.colab import files
files.download('reward_curve_real.png')

print(f"\n📊 W&B Run: {run.url}")
print("\n✅ Download complete! Next: Update repo with real results")
